# Population Aerotaxis — temporal analysis

Behavioural state (speed, reversals, turns, body-bend frequency) locked to
**global gas shifts**. Works on the tidy table from `create_results_dict_server.py`
(`aerotaxis_results.{parquet,csv,pkl}`) with columns:
`Condition, Recording, Crop_ID, Frame, Time_Seconds, O2_State, Forward_Velocity,
Reversal_Active, Turn_Active, Reversal_Onset, Bend_Frequency, Bend_Amplitude`.

`Frame`/`Time_Seconds` are ABSOLUTE (recording-wide), so crops that start at
different times all share one clock. Analysis primitives live in
`toolscripts/utils/aerotaxis_analysis.py`.

### 1. Load the tidy results

In [ ]:
import sys
from pathlib import Path

UTILS = Path.cwd().parents[1] / 'toolscripts' / 'utils'
if UTILS.exists():
    sys.path.insert(0, str(UTILS))
import aerotaxis_analysis as aa

RESULTS = 'aerotaxis_results.parquet'   # or a dataset folder
df = aa.load_results(RESULTS)
print(f'{len(df):,} rows | states: {sorted(df.O2_State.unique())}')
df.head()

### 2. Per-state summary
Rates (reversal/turn fraction, reversal onsets/min) and means (speed, bend Hz) per gas state.

In [ ]:
summary = aa.per_state_summary(df)
summary

### 3. Behaviour by gas state (per condition)

In [ ]:
import matplotlib.pyplot as plt
metrics = [m for m in ['mean_forward_velocity','reversal_fraction','turn_fraction',
                       'mean_bend_frequency_hz','reversal_onsets_per_min'] if m in summary]
fig, axes = plt.subplots(1, len(metrics), figsize=(4*len(metrics), 4))
for ax, m in zip(axes, metrics):
    aa.plot_per_state_summary(summary, metric=m, ax=ax)
fig.tight_layout()

### 4. Gas-transition-triggered averages
Align a feature to every O2 shift (t=0 at the shift), averaged across crops.
Swap `feature` for any continuous column: `Forward_Velocity`, `Bend_Frequency`,
`Reversal_Active`, `Turn_Active`.

In [ ]:
for feat in ['Forward_Velocity', 'Bend_Frequency', 'Reversal_Active']:
    if feat in df and df[feat].notna().any():
        tta = aa.transition_triggered_average(df, feature=feat, pre_s=10, post_s=30)  # fps read from the data's fps column
        fig, ax = plt.subplots(figsize=(9, 3.5))
        aa.plot_transition_triggered(tta, feature=feat, ax=ax)
        plt.show()

### 5. Reversal reaction to the O2 pulse
Latency from each pulse onset to the first reversal (generalises `rev_reaction.py`).

In [ ]:
rr = aa.reversal_reaction(df, to_state='21pct_O2', window_s=15)  # fps read from the data's fps column
print(f"{rr.reacted.mean()*100:.0f}% of pulses triggered a reversal within 15 s; "
      f"median latency {rr.latency_s.median():.2f} s")
import seaborn as sns
sns.histplot(rr.dropna(subset=['latency_s']), x='latency_s', hue='Condition', bins=30)
plt.xlabel('reversal latency after pulse onset (s)'); plt.show()

### 6. Per-crop time-series viewer
Interactive: pick a crop; features with the gas protocol shaded.

In [ ]:
import ipywidgets as widgets
import numpy as np

df['_crop'] = df[aa.GROUP_KEYS].agg(' / '.join, axis=1)
crops = sorted(df['_crop'].unique())
states = sorted(df.O2_State.unique())
cmap = {s: c for s, c in zip(states, plt.cm.tab10.colors)}
panels = [('Forward_Velocity','fwd vel\n(mm/s)'), ('Bend_Frequency','bend\n(Hz)'),
          ('Reversal_Active','reversal'), ('Turn_Active','turn')]
panels = [(c,l) for c,l in panels if c in df]

def show(crop):
    sub = df[df['_crop']==crop].sort_values('Frame'); t = sub.Time_Seconds.to_numpy()
    fig, ax = plt.subplots(len(panels), 1, figsize=(12, 2*len(panels)), sharex=True)
    for a,(col,lab) in zip(ax, panels):
        a.plot(t, sub[col]); a.set_ylabel(lab)
    st = sub.O2_State.to_numpy()
    edges = [0]+list(np.where(st[1:]!=st[:-1])[0]+1)+[len(st)]
    for a in ax:
        for i in range(len(edges)-1):
            lo, hi = edges[i], edges[i+1]-1
            a.axvspan(t[lo], t[hi], color=cmap.get(st[lo],'gray'), alpha=0.12)
    ax[0].set_title(crop); ax[-1].set_xlabel('recording time (s)')
    fig.tight_layout(); plt.show()

widgets.interact(show, crop=widgets.Dropdown(options=crops, description='crop'));

### 7. Save tidy summaries (for R / sharing)

In [ ]:
outdir = Path('analysis'); outdir.mkdir(exist_ok=True)
summary.to_csv(outdir/'per_state_summary.csv', index=False)
rr.to_csv(outdir/'reversal_reaction.csv', index=False)
print('wrote', [p.name for p in outdir.glob('*.csv')])